# Size does not matter: Sub-Billion VLM NanoChimera is All You Need!

This project addresses the modern challenge of Vision-Language Model (VLM) deployment by testing the scalability hypothesis: **Is massive parameter count necessary for effective visual reasoning?**

We construct and train **NanoChimera VLM**, a novel modular architecture designed to achieve high performance while strictly remaining **Sub-Billion** parameters ($\approx 999.99 \text{ Million}$ total deployed). Our goal is threefold: to provide a practical tutorial on building custom VLM architectures by training the critical Projector Layer, to rigorously evaluate its cognitive capabilities (grounding and hallucination), and to serve as a proof-of-concept for **real-time, edge-friendly multimodal AI**.


First of all, lets import the libraries and set a common seed 42 so the experiment is reproducible.


The outline of the notebook goes as following:

0) Data Loading and Augmentations
1) Model Architecture
2) Training Pipeline
3) Evaluation Pipeline
4) Experiments
5) Results
6) Conclusions

In [ ]:
# ============================================================
# EXECUTION CONTROL FLAGS
# ============================================================
# Toggle these to control what runs on "Run All".
# Everything above the flags (imports, definitions) always executes.
# ============================================================

import os, json
import torch

RUNNING_ON_GOOGLE = True         # True if running on Google Colab
OVERWRITE_DATASET = True
PRETRAINED_MODEL_PATH = "./runs/run_20260202_0311_ds8192_bs64_h4096_ep8"
PRETRAINED_MODEL_FILE = "best_nano_chimera.pt"
RESULTS_JSON = "results.json"

# --- pipeline stages ---
PERFORM_STAGE1_TRAINING  = True   # Stage 1: vision-language alignment pretraining
PERFORM_INFERENCE       = True   # Run inference on test set after Stage 1
PERFORM_VISFT           = False  # Stage 2: Visual Instruction Fine-Tuning (SFT)
PERFORM_EVALUATION       = True   # LMMS benchmark evaluation
PERFORM_EXPERIMENTS      = False  # Sweep over EXPERIMENTS grid (see below)
PUSH_TO_HUB              = False  # Push final model to HuggingFace

# --- experiment grid (only used when PERFORM_EXPERIMENTS = True) ---
# Each dict is a full training config. The loop rebuilds the connector
# per entry so hidden_dims / n_visual_tokens changes take effect.
EXPERIMENTS = [
    {   # 3.1 — single-layer baseline
        "name":                   "1-layer-4096",
        "dataset_size":           10000,
        "batch_size":             16,
        "learning_rate":          1e-4,
        "weight_decay":           0.01,
        "n_visual_tokens":        32,
        "connector_hidden_dims":  (4096,),
        "epochs":                 4,
        "grad_accum_steps":       8,
        "warmup_ratio":           0.05,
        "max_grad_norm":          1.0,
        "scheduler_start_factor": 0.1,
        "log_every":              32,
        "eval_every":             256,
        "label_smoothing":        0.05,
    },
    {   # 3.2 — two-layer adapter
        "name":                   "2-layer-4096x2048",
        "dataset_size":           10000,
        "batch_size":             16,
        "learning_rate":          1e-4,
        "weight_decay":           0.01,
        "n_visual_tokens":        32,
        "connector_hidden_dims":  (4096, 2048),
        "epochs":                 4,
        "grad_accum_steps":       8,
        "warmup_ratio":           0.05,
        "max_grad_norm":          1.0,
        "scheduler_start_factor": 0.1,
        "log_every":              32,
        "eval_every":             256,
        "label_smoothing":        0.05,
    },
]

if not PERFORM_STAGE1_TRAINING:
    PRETRAINED_MODEL_NAME = os.path.join(PRETRAINED_MODEL_PATH, PRETRAINED_MODEL_FILE)
    try:
        nano_chimera = torch.load(PRETRAINED_MODEL_NAME, map_location="cpu")
    except Exception as e:
        raise RuntimeError(
            f"Failed to load nano_chimera from {PRETRAINED_MODEL_NAME}"
        ) from e
    RESULTS_PATH = os.path.join(PRETRAINED_MODEL_PATH, RESULTS_JSON)
    if os.path.isfile(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            EX_CONFIG = json.load(f).get("config", {})
            print("Loaded config from results.json:", EX_CONFIG)
    else:
        raise RuntimeError(f"Could not find results.json at {results_path}")
# ============================================================
# IMPORTS & SEED
# ============================================================
import time, pickle, random, math
from datetime import datetime
from typing import *

if RUNNING_ON_GOOGLE:
    from google.colab import drive, userdata
    os.environ["HUGGINGFACE_TOKEN"] = userdata.get("HUGGINGFACE_TOKEN")
    os.environ["HF_TOKEN"]          = userdata.get("HF_TOKEN")
    os.environ["OPENAI_API_KEY"]    = userdata.get("OPENAI_API_KEY")
    %pip install loguru
    %pip install lmms-eval

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from loguru import logger
from tqdm import tqdm

import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader, random_split

from lmms_eval.api.instance import Instance
from lmms_eval.models import get_model
from lmms_eval.tasks import TaskManager, get_task_dict
from lmms_eval.api.model import lmms
from lmms_eval.evaluator import simple_evaluate

from datasets import load_dataset
import huggingface_hub
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel

def set_seed(seed, use_gpu=True, change_numpy_seed=False):
    random.seed(seed)
    if change_numpy_seed:
        np.random.seed(seed)
    torch.manual_seed(seed)
    if use_gpu:
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED, torch.cuda.is_available())

huggingface_hub.login(token=os.environ.get("HUGGINGFACE_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 0. Data Loading & Augmentations

Thanks to the already well curated efforts by meta with the LLaVa model, we do not need to preprocess the data at all, its just a plug and play clean dataset. Now it has to be said that if we wanted to artificially augment the size of the dataset this could easily be done throught common computer vision data augmentation techniques, however due to the sheer data volume we dispose. Nevertheless, for completeness, below we add a set of transformations that could be easily plugged into the Torch dataset API to perform data augmentation.

In [ ]:
class VLM_Dataset(Dataset):
    def __init__(
        self,
        size=10000,
        filename="training_dataset.pkl",
        dataset="damerajee/Llava-pretrain-small",
        split="train",
        overwrite=False,
        seed=42,
        transform=None
    ):
        self.transform = transform

        # Load from pickle if available
        if os.path.exists(filename) and not overwrite:
            with open(filename, "rb") as f:
                self.data = pickle.load(f)
            return

        # Otherwise, read dataset from HuggingFace
        stream = load_dataset(
            dataset,
            split=split,
            streaming=True
        ).shuffle(buffer_size=size, seed=seed)

        self.data = []
        for i, row in enumerate(stream):
            if i >= size:
                break

            # Store the raw caption — the training loop owns the <image> token
            self.data.append({
                "image": row["image"],
                "caption": row["answer"]
            })

        # Save dataset for future use
        with open(filename, "wb") as f:
            pickle.dump(self.data, f)

    # ---------- PyTorch API ----------
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if self.transform:
            item["image"] = self.transform(item["image"])
        return item

In [ ]:
# Data augmentation transformations
"""
transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(p=0.3),
    torchvision.transforms.RandomVerticalFlip(p=0.3),
    torchvision.transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.05),
    torchvision.transforms.RandomGrayscale(p=0.1),
    torchvision.transforms.RandomAdjustSharpness(1.5, p=0.2),
    torchvision.transforms.ToTensor(),
])

# Basic transformation
transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.ToTensor(),
])
"""

'\ntransforms = torchvision.transforms.Compose([\n    torchvision.transforms.Resize(224),\n    torchvision.transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),\n    torchvision.transforms.RandomHorizontalFlip(p=0.3),\n    torchvision.transforms.RandomVerticalFlip(p=0.3),\n    torchvision.transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.05),\n    torchvision.transforms.RandomGrayscale(p=0.1),\n    torchvision.transforms.RandomAdjustSharpness(1.5, p=0.2),\n    torchvision.transforms.ToTensor(),\n])\n\n# Basic transformation\ntransforms = torchvision.transforms.Compose([\n    torchvision.transforms.Resize(224),\n    torchvision.transforms.ToTensor(),\n])\n'

In [ ]:
# Example of applying transformations to K images
"""
dataset = VLM_Dataset(
    size=50,
    transform=transforms
)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


# Get a batch
batch = next(iter(loader))
images = batch["image"]
captions = batch["caption"]
K = images.size(0)
cols = 4
rows = math.ceil(K / cols)

plt.figure(figsize=(cols * 4, rows * 4))

for i in range(K):
    img = images[i].permute(1, 2, 0)

    # If normalized, undo it (adjust if you used different stats)
    img = img * 0.5 + 0.5
    img = img.clamp(0, 1)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(captions[i][:40], fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

"""
None

## 1. Model Architecture

This is one of the single most important points of this project, the architecture of NanoChimera that will allow us to connect visual to textual tokens and thus allow the LLM to understand images. Therefore the visual adapter and its quality is the biggest factor determining the quality of this VLM.

The easiest implementation of this adapter is an MLP, and we can also relax the implicit assumption that all the image vision tokens should be used through the connector and passed down to the LLM, which is often not the case, some visual tokens have undoubtedly more importance than others.

In [ ]:
class VisionConnector(nn.Module):
    """
    Maps vision encoder features -> LLM embedding space
    Supports a flexible number of hidden layers.
    """
    def __init__(self, vision_dim, llm_dim, hidden_dims=(4096,), device = None):
        super().__init__()
        self.device = device

        layers = []
        # Start with vision_dim
        current_dim = vision_dim

        # Add hidden layers dynamically
        for h_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.GELU())
            current_dim = h_dim

        # Add the final projection layer to llm_dim
        layers.append(nn.Linear(current_dim, llm_dim))

        self.proj = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, vision_feats):
        # vision_feats: (B, N, vision_dim)
        return self.proj(vision_feats)  # (B, N, llm_dim)

In [ ]:
def merge_text_and_visual_embeddings(
    images,
    K,
    device,
    llm,
    tokenizer,
    connector,
    texts=None
):
    B = images.size(0)

    if texts is None:
        texts = ["<image> Describe this image in one sentence."] * B

    # 1. Tokenize
    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    input_ids = encoded.input_ids.to(device)          # [B, L]
    attention_mask_text = encoded.attention_mask.to(device)
    text_embeds = llm.get_input_embeddings()(input_ids)  # [B, L, D]

    # 2. Encode images
    with torch.no_grad():
        vision_feats = vision_model.vision_model(pixel_values=images.to(device)).last_hidden_state
    vis_embeds = connector(vision_feats)[:, :K, :]  # [B, K, D]

    # 3. Find <image> token positions
    image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
    image_token_idx = (input_ids == image_token_id).nonzero(as_tuple=True)[1]  # [B]

    # 4. Compute final sequence length
    L_text = input_ids.size(1)
    L_final = L_text + K - 1  # remove 1 token where <image> was
    D = text_embeds.size(2)

    # 5. Preallocate merged tensors
    inputs_embeds = torch.zeros(B, L_final, D, device=device)
    attention_mask = torch.zeros(B, L_final, device=device)

    # 6. Fill merged embeddings in batch
    for b in range(B):
        idx = image_token_idx[b].item()
        inputs_embeds[b, :idx] = text_embeds[b, :idx]
        inputs_embeds[b, idx:idx+K] = vis_embeds[b]
        inputs_embeds[b, idx+K:] = text_embeds[b, idx+1:]

        attention_mask[b, :idx+K] = 1
        attention_mask[b, idx+K:] = 1

    return inputs_embeds, attention_mask, image_token_idx.tolist()

In [ ]:
class NanoChimera(nn.Module):
    """
    LLaVA-style VLM:
    - Frozen vision encoder
    - Frozen LLM
    - Trainable connector
    """

    def __init__(self, vision_encoder, llm, connector):
        super().__init__()
        self.vision = vision_encoder
        self.llm = llm
        self.connector = connector

        self.vision.requires_grad_(False)
        self.llm.requires_grad_(False)

    def forward(
        self,
        images,
        input_ids,
        attention_mask,
        image_token_id,
        labels=None,
    ):
        # ---- vision ----
        with torch.no_grad():
            vision_out = self.vision(images)
            vision_feats = vision_out.last_hidden_state  # (B, N, vision_dim)

        # ---- connector ----
        visual_tokens = self.connector(vision_feats)  # (B, N, llm_dim)

        # ---- build inputs ----
        if labels is not None:
            inputs_embeds, attn_mask, labels = build_inputs(
                llm=self.llm,
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_tokens=visual_tokens,
                image_token_id=image_token_id,
                labels=labels,
            )
        else:
            inputs_embeds, attn_mask = build_inputs(
                llm=self.llm,
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_tokens=visual_tokens,
                image_token_id=image_token_id,
            )

        # ---- LLM ----
        return self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask,
            labels=labels,
        )

## 2. Training Pipeline definition

Now we will define a simple data splitting (train/validation/test) strategy along the training loop, to add some interesting MLOps features that will improve the quality of the training experience, we will add logging, persistent information of runs and managing checkpoints to always save the best models comparing to past models.

So the section is split into the following subsections:

1) Data splitting
2) Model setup
3) Discussion on simple training evaluation metrics
4) Training loop definition
5) Training example (**Pretraining**)
6) Pipeline extension (**Supervised Fine-Tuning**)




Below we can find a simple example of a pipeline hyperparameters configuration

In [ ]:
if PERFORM_STAGE1_TRAINING:
    # This is a sample
    EX_CONFIG = {
      # Data
      #"dataset_size": 16384,
      "dataset_size": 74000,
      "batch_size": 58,
      #"batch_size": 256,

      # Optimization algorithm
      "learning_rate": 2e-5,
      "weight_decay": 0.005,

      # Connector parameters
      "n_visual_tokens": 32,
      "connector_hidden_dims": (4096, ),

      # training
      "epochs": 8,
      "grad_accum_steps": 8,
      "warmup_ratio": 0.1,
      "max_grad_norm": 0.8,
      "scheduler_start_factor": 0.05,

      # logging
      "log_every": 32,
      "eval_every": 256,

      # criterion
      "label_smoothing": 0.02,
    }

### 2.1 Splitting Strategy and Data Preparation

For now we won't make use of automatic cross validation and rely on a simple train/test/validation split. Thanks to Pytorch data loaders we can shuffle when sampling to avoid overfitting weird patterns.


In [9]:
def split_dataset(dataset, train=0.9, val=0.05, test=0.05, seed=SEED):
    # Check for correct splitting
    assert train + val + test == 1.0
    # Getting lengths of datasets
    n = len(dataset)
    n_train = int(train * n)
    n_val = int(val * n)
    n_test = n - n_train - n_val
    # Using seed
    generator = torch.Generator().manual_seed(seed)

    return random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )

# For Pytorch Dataloader API
def collate_fn(batch, img_size=(224, 224)):
    # Keep PIL images for the processor and unifying pictures into one format
    img_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(img_size),
        torchvision.transforms.Lambda(lambda img: img.convert("RGB")),  # force 3 channels
        torchvision.transforms.ToTensor()
    ])

    images = torch.stack([img_transform(item["image"]) for item in batch])
    captions = [item["caption"] for item in batch]
    return {"image": images, "caption": captions}

# LOAD DATA
data = VLM_Dataset(size=EX_CONFIG["dataset_size"], overwrite=OVERWRITE_DATASET)
train_dataset, val_dataset, test_dataset = split_dataset(data)
if PERFORM_STAGE1_TRAINING:
    train_dataset, val_dataset, test_dataset = split_dataset(data)

    train_loader = DataLoader(
        train_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
    )
else:
    _, _, test_dataset = split_dataset(data)

test_loader = DataLoader(
    test_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)

README.md:   0%|          | 0.00/364 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

### 2.2 Model setup

In [ ]:
# LOAD MODELS, LLM TOKENIZER, VISION PROCESSOR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip2-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.float32
).to(DEVICE)

vision_processor = AutoProcessor.from_pretrained(VISION_NAME, use_fast=True)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, dtype=torch.float32
).to(DEVICE)

# ADD IMAGE TOKEN to LLM tokenizer vocabulary so it recognizes images
IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# CONSTRUCT NANO CHIMERA MODEL (QWEN 2.5 + SIGILP + CONNECTOR)
# SigLIP Embedding output dimension
vision_dim = vision_model.config.vision_config.hidden_size
# LLM Embedding output dimension
llm_dim = llm.config.hidden_size

connector = VisionConnector(
    vision_dim=vision_dim,
    llm_dim=llm_dim,
    hidden_dims=EX_CONFIG["connector_hidden_dims"],
    device=DEVICE
).to(DEVICE).float()

if PERFORM_STAGE1_TRAINING:
    nano_chimera = NanoChimera(
        vision_encoder=vision_model.vision_model,
        llm=llm,
        connector=connector
    ).to(DEVICE)

NanoChimera

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

### 2.3 Discussion on simple training evaluation metrics

As we already know LLM evaluation is a non-trivial task due to all the nuances of language, then multimodal LLM (MLLM) being a strict superset of these models, we can easily see the complexity grows as the number of modalities grows, strictly. Real performance evaluation is usually done through benchmarks and human/llm-as-a-judge evaluation, although for training such expensive evaluation metrics cannot be used.


So besides **negative log-likelihood (NLL) loss** or **cross-entropy loss** for raw token classification, we would benefit from an interpretable metric to evaluate (similar to accuracy, recall or f1), which for LLMs pretraining its usually either of these 2:

1) Token-Accuracy: measures the fraction of correctly predicted tokens, ignoring masked tokens (e.g. image tokens or padding), in other words: **Did the model’s argmax token match the label?**. Its easy to interpret but satures quickly and ignores near-misses and confidence!


$
\text{Token Accuracy}
=
\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\mathbf{1}\!\left[ \hat{y}_t = y_t \right]
$

2) Perplexity: the exponential of the average negative log-likelihood. In other words: **On average, how many equally likely tokens the model is confused between.** When the PPL = 1, then the prediction is perfect, otherwise PPL = 5 its a random guess between 5 tokens, the lower the better.

$
\text{Perplexity}
=
\exp\!\left(
\mathcal{L}_{\mathrm{NLL}}
\right)
=
\exp\!\left(
-\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\log p_\theta\!\left( y_t \mid x_{<t} \right)
\right)
$

So in conclusion, Token Accuracy can be used for sanity check and debugging, but to get an actual intuition of performance north-star we will use perplexity.

In [ ]:
class LMStats:
    """Accumulates language modeling statistics."""
    def __init__(self):
        self.loss_sum = 0.0
        self.token_count = 0
        self.correct = 0

    def update(self, loss, logits, labels):
        """
        loss: scalar CE loss (already reduced)
        logits: [1, T, V]
        labels: [1, T] with -100 masked tokens
        """
        mask = labels != -100
        n_tokens = mask.sum().item()

        self.loss_sum += loss.item() * n_tokens
        self.token_count += n_tokens

        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            self.correct += ((preds == labels) & mask).sum().item()

    def avg_loss(self):
        return self.loss_sum / max(self.token_count, 1)

    def perplexity(self):
        return math.exp(self.avg_loss())

    def accuracy(self):
        return self.correct / max(self.token_count, 1)

### 2.4 Training Pipeline definition

BIG TODO: The pipeline efficiceny can still be greatly improved, specially the auxilliary functions like build inputs and others which are not batched!

In [ ]:
def train_one_epoch(model,
                    loader,
                    optimizer,
                    scheduler,
                    connector,
                    criterion,
                    device,
                    EX_CONFIG,
                    warmup_steps,
                    global_step,
                    debug=False):
    K = EX_CONFIG["n_visual_tokens"]
    model.train()
    total_loss, total_acc, total_tokens = 0.0, 0.0, 0
    pbar = tqdm(loader, desc="Training", unit="batch")

    image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)

    for batch in pbar:
        imgs = batch["image"].to(device)
        captions = batch["caption"]  # raw captions, no <image> prefix

        # --------------------------------------------------------
        # 1. Build full text: prompt + caption as one sequence
        #    Layout in input_ids:
        #      [ <image> ] [ prompt tokens ] [ caption tokens ]
        #    After merge (replacing first <image> with K visual tokens):
        #      [ v0 … v_{K-1} ] [ prompt tokens ] [ caption tokens ]
        # --------------------------------------------------------
        question_input = "<image> What is in the picture?"
        full_texts = [f"{question_input} {cap}" for cap in captions]

        tokenized = tokenizer(
            full_texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        input_ids = tokenized.input_ids.to(device)

        # --------------------------------------------------------
        # 2. Merge image embeddings into the sequence
        # --------------------------------------------------------
        inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
            images=imgs,
            K=K,
            device=device,
            llm=model.llm,
            tokenizer=tokenizer,
            connector=connector,
            texts=full_texts
        )

        B, L, D = inputs_embeds.size()

        # --------------------------------------------------------
        # 3. Build labels — ONLY caption tokens are targets
        #
        #    We need to find where the caption starts in input_ids.
        #    The prompt is "<image> What is in the picture? "
        #    so we tokenize JUST the prompt to measure its length,
        #    then everything after that in input_ids is caption.
        #
        #    In the MERGED sequence the first <image> token has been
        #    replaced by K visual tokens, so all positions after it
        #    shift by (K - 1).  Caption starts at:
        #        merged_caption_start = prompt_token_len - 1 + K
        #    (prompt_token_len includes the <image> token itself)
        # --------------------------------------------------------
        prompt_len = tokenizer(
            question_input,
            return_tensors="pt",
            padding=False,
            truncation=True
        ).input_ids.size(1)  # number of tokens in the prompt alone

        labels = input_ids.new_full((B, L), -100)

        for b in range(B):
            # Where caption tokens start in the MERGED sequence
            merged_caption_start = prompt_len - 1 + K  # -1 for the removed <image>, +K for visual tokens

            # Caption token IDs in the original input_ids
            caption_ids = input_ids[b, prompt_len:]   # everything after the prompt
            n_cap = caption_ids.size(0)

            # Place them into labels at the correct merged positions
            end = merged_caption_start + n_cap
            if end > L:
                end = L
                n_cap = end - merged_caption_start
            labels[b, merged_caption_start:end] = caption_ids[:n_cap]

        # --------------------------------------------------------
        # DEBUG: token / label alignment sanity check
        # --------------------------------------------------------
        if global_step == 0 and debug:
            b = 0
            print("\n================ DEBUG SAMPLE ================")
            print("FULL TEXT:")
            print(full_texts[b])

            print("\nTOKENIZED INPUT (original input_ids):")
            print(tokenizer.convert_ids_to_tokens(input_ids[b]))

            print(f"\nIMAGE TOKEN IDX: {image_token_idx[b]}")
            print(f"K (visual tokens): {K}")
            print(f"Prompt token count: {prompt_len}")
            print(f"Merged caption start: {merged_caption_start}")

            print("\nLABELED POSITIONS (non -100):")
            labeled_positions = (labels[b] != -100).nonzero(as_tuple=True)[0]
            for pos in labeled_positions[:30]:
                tok_id = labels[b, pos].item()
                print(f"  {pos.item()}: {tokenizer.convert_ids_to_tokens(tok_id)}")

            print("\nSANITY CHECK:")
            print("--> No prompt tokens ('What','is','in',...) should appear above")
            print("--> No <image> token should appear above")
            print("--> First labeled token should be the first caption word")

        # --------------------------------------------------------
        # 4. Forward pass
        # --------------------------------------------------------
        outputs = model.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=None  # manual loss
        )
        logits = outputs.logits  # [B, L, vocab_size]

        # --------------------------------------------------------
        # 5. Compute loss on shifted logits/labels
        # --------------------------------------------------------
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        raw_loss = criterion(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

        # --------------------------------------------------------
        # 6. Backward pass + optimizer step
        # --------------------------------------------------------
        (raw_loss / EX_CONFIG["grad_accum_steps"]).backward()

        if (global_step + 1) % EX_CONFIG["grad_accum_steps"] == 0:
            optimizer.step()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

        # --------------------------------------------------------
        # 7. Accumulate metrics (Bug E fix: weight loss by token count)
        # --------------------------------------------------------
        n_valid = (shift_labels != -100).sum().item()
        total_loss  += raw_loss.item() * n_valid   # weight by tokens, not batch size
        total_tokens += n_valid

        preds   = shift_logits.argmax(dim=-1)
        correct = ((preds == shift_labels) & (shift_labels != -100)).sum().item()
        total_acc += correct

        avg_loss = total_loss / max(total_tokens, 1)
        avg_ppl  = math.exp(avg_loss)
        avg_acc  = total_acc / max(total_tokens, 1)

        pbar.set_postfix({
            "acc":    f"{avg_acc:.4f}",
            "loss":   f"{avg_loss:.4f}",
            "ppl":    f"{avg_ppl:.2f}",
            "tokens": total_tokens
        })

        global_step += 1

    avg_loss = total_loss / max(total_tokens, 1)
    avg_acc  = total_acc  / max(total_tokens, 1)
    avg_ppl  = math.exp(avg_loss)

    return avg_loss, avg_ppl, avg_acc, total_tokens, global_step

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, EX_CONFIG):
    K = EX_CONFIG["n_visual_tokens"]
    model.eval()
    total_loss, total_acc, total_tokens = 0.0, 0.0, 0
    pbar = tqdm(loader, desc="Evaluating", unit="batch")

    question_input = "<image> What is in the picture?"
    prompt_len = tokenizer(
        question_input,
        return_tensors="pt",
        padding=False,
        truncation=True
    ).input_ids.size(1)

    with torch.no_grad():
        for batch in pbar:
            imgs     = batch["image"].to(device)
            captions = batch["caption"]  # raw captions, no <image> prefix

            # 1. Build full text exactly as in training
            full_texts = [f"{question_input} {cap}" for cap in captions]

            tokenized = tokenizer(
                full_texts,
                return_tensors="pt",
                padding=True,
                truncation=True
            )
            input_ids = tokenized.input_ids.to(device)

            # 2. Merge embeddings
            inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
                images=imgs,
                K=K,
                device=device,
                llm=model.llm,
                tokenizer=tokenizer,
                connector=connector,
                texts=full_texts
            )

            B, L, D = inputs_embeds.size()

            # 3. Build labels - identical logic to training
            labels = input_ids.new_full((B, L), -100)
            for b in range(B):
                merged_caption_start = prompt_len - 1 + K
                caption_ids = input_ids[b, prompt_len:]
                n_cap = caption_ids.size(0)
                end = merged_caption_start + n_cap
                if end > L:
                    end = L
                    n_cap = end - merged_caption_start
                labels[b, merged_caption_start:end] = caption_ids[:n_cap]

            # 4. Forward pass
            outputs = model.llm(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                labels=None
            )
            logits = outputs.logits  # [B, L, vocab_size]

            # 5. Shifted loss
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            raw_loss = criterion(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )

            # 6. Accumulate metrics (weighted by valid token count)
            n_valid = (shift_labels != -100).sum().item()
            total_loss   += raw_loss.item() * n_valid
            total_tokens += n_valid

            preds   = shift_logits.argmax(dim=-1)
            correct = ((preds == shift_labels) & (shift_labels != -100)).sum().item()
            total_acc += correct

            avg_loss = total_loss / max(total_tokens, 1)
            avg_ppl  = math.exp(avg_loss)
            avg_acc  = total_acc / max(total_tokens, 1)

            pbar.set_postfix({
                "acc":    f"{avg_acc:.4f}",
                "loss":   f"{avg_loss:.4f}",
                "ppl":    f"{avg_ppl:.2f}",
                "tokens": total_tokens
            })

    avg_loss = total_loss / max(total_tokens, 1)
    avg_ppl  = math.exp(avg_loss)
    avg_acc  = total_acc / max(total_tokens, 1)

    return avg_loss, avg_ppl, avg_acc, total_tokens

In [ ]:
def model_training(
    n_epochs,
    nano_chimera,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    scheduler,
    warmup_steps,
    model_name="best_nano_chimera.pt"
):
    """
    Standard training loop for multimodal LLMs.
    - Tracks loss, perplexity, token accuracy, token counts
    - Saves best checkpoint
    - Saves JSON + PNG + PDF report for each run
    """

    # 0. Experiment naming & folders
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    dataset_size = EX_CONFIG["dataset_size"]
    batch_size   = EX_CONFIG["batch_size"]
    hidden = "x".join(map(str, EX_CONFIG["connector_hidden_dims"]))
    run_name = f"run_{timestamp}_ds{dataset_size}_bs{batch_size}_h{hidden}_ep{n_epochs}"

    output_dir = os.path.join("runs", run_name)
    os.makedirs(output_dir, exist_ok=True)

    best_model_path = os.path.join(output_dir, model_name)
    last_model_path = os.path.join(output_dir, "last_checkpoint.pt")
    json_path  = os.path.join(output_dir, "results.json")
    plot_path  = os.path.join(output_dir, "training_metrics.pdf")
    png_path   = os.path.join(output_dir, "training_metrics.png")
    pdf_path   = os.path.join(output_dir, "report.pdf")

    # 1. Training bookkeeping
    best_val_loss = float("inf")

    train_losses, train_accs, train_ppls = [], [], []
    val_losses,   val_accs,   val_ppls   = [], [], []
    train_token_counts, val_token_counts = [], []

    global_step = 0

    # NEW: interruption tracking
    interrupted = False
    last_epoch = -1

    # 2. Training loop
    try:
        for epoch in range(n_epochs):
            start = time.time()
            last_epoch = epoch

            train_loss, train_ppl, train_acc, train_tokens, global_step = train_one_epoch(
                nano_chimera,
                train_loader,
                optimizer,
                scheduler,
                connector,
                criterion,
                device,
                EX_CONFIG,
                warmup_steps,
                global_step
            )

            val_loss, val_ppl, val_acc, val_tokens = evaluate(
                nano_chimera,
                val_loader,
                criterion,
                device,
                EX_CONFIG
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(nano_chimera.state_dict(), best_model_path)

            logger.info(
                f"Epoch {epoch+1}/{n_epochs} | "
                f"Train Loss {train_loss:.4f} | PPL {train_ppl:.2f} | Acc {train_acc:.4f} | Tokens {train_tokens} | "
                f"Val Loss {val_loss:.4f} | PPL {val_ppl:.2f} | Acc {val_acc:.4f} | Tokens {val_tokens} | "
                f"Time {time.time()-start:.1f}s"
            )

            train_losses.append(train_loss)
            train_accs.append(train_acc)
            train_ppls.append(train_ppl)
            train_token_counts.append(train_tokens)

            val_losses.append(val_loss)
            val_accs.append(val_acc)
            val_ppls.append(val_ppl)
            val_token_counts.append(val_tokens)

    except KeyboardInterrupt:
        interrupted = True
        logger.warning(
            f"Training interrupted by user at epoch {last_epoch+1}/{n_epochs}. "
            "Saving partial results."
        )

    finally:
        # Always save last checkpoint (even on Ctrl+C)
        torch.save(nano_chimera.state_dict(), last_model_path)

    # Guard: if nothing ran, skip reporting
    if len(train_losses) == 0:
        logger.error("No epochs completed — skipping report generation.")
        return None

    # 3. Plot metrics
    train_token_cum = np.cumsum(train_token_counts)
    val_token_cum   = np.cumsum(val_token_counts)

    plots = [
        (train_losses, val_losses, "Loss"),
        (train_accs,   val_accs,   "Token Accuracy"),
        (train_token_cum, val_token_cum, "Cumulative Tokens"),
        (train_ppls,   val_ppls,   "Perplexity")
    ]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, (train_data, val_data, title) in zip(axes, plots):
        ax.plot(train_data, label="Train")
        ax.plot(val_data,   label="Val")
        ax.set_title(title)
        ax.grid(True)
        ax.legend()

    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.savefig(png_path,  dpi=150)
    plt.close()

    # 4. Save RUN Report JSON
    run_summary = {
        "run_name":  run_name,
        "timestamp": timestamp,
        "config":    EX_CONFIG,
        "interrupted": interrupted,
        "completed_epochs": last_epoch + 1,
        "metrics": {
            "train_loss":               train_losses,
            "val_loss":                 val_losses,
            "train_accuracy":           train_accs,
            "val_accuracy":             val_accs,
            "train_perplexity":         train_ppls,
            "val_perplexity":           val_ppls,
            "train_tokens_epoch":       train_token_counts,
            "val_tokens_epoch":         val_token_counts,
            "train_tokens_cumulative":  train_token_cum.tolist(),
            "val_tokens_cumulative":    val_token_cum.tolist(),
        },
        "artifacts": {
            "plot":         plot_path,
            "best_model":   best_model_path,
            "last_model":   last_model_path,
            "pdf_report":   pdf_path
        },
        "best_val_loss": best_val_loss
    }

    with open(json_path, "w") as f:
        json.dump(run_summary, f, indent=2)

    # 5. Generate PDF report
    with PdfPages(pdf_path) as pdf:
        fig = plt.figure(figsize=(11, 8))

        plt.text(0.01, 0.95, "Final Metrics Summary", fontsize=14)

        summary_lines = [
            f"Final Train Loss: {train_losses[-1]:.4f}",
            f"Final Val Loss:   {val_losses[-1]:.4f}",
            f"Final Train Acc:  {train_accs[-1]:.4f}",
            f"Final Val Acc:    {val_accs[-1]:.4f}",
            f"Final Train PPL:  {train_ppls[-1]:.2f}",
            f"Final Val PPL:    {val_ppls[-1]:.2f}",
            "",
            f"Total Train Tokens: {int(train_token_cum[-1])}",
            f"Total Val Tokens:   {int(val_token_cum[-1])}",
            "",
            f"Best Validation Loss: {best_val_loss:.4f}",
            "",
            f"Interrupted: {interrupted}",
            f"Completed Epochs: {last_epoch + 1}"
        ]

        y = 0.88
        for line in summary_lines:
            plt.text(0.01, y, line, fontsize=11)
            y -= 0.05

        y = 0.85
        for k, v in EX_CONFIG.items():
            plt.text(0.01, y, f"{k}: {v}", fontsize=9)
            y -= 0.03

        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

        img = plt.imread(png_path)
        fig = plt.figure(figsize=(11, 5))
        plt.imshow(img)
        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

    logger.info(f"Run saved to {output_dir}")

    return (
        train_losses,
        train_accs,
        val_losses,
        val_accs,
        train_token_counts,
        val_token_counts
    )


### 2.5 Example (Train + Inference)


In [ ]:
if PERFORM_STAGE1_TRAINING:
    criterion = nn.CrossEntropyLoss(ignore_index=-100,
        label_smoothing=EX_CONFIG["label_smoothing"])
    criterion = criterion.to(DEVICE)

    optimizer = torch.optim.AdamW(
        connector.parameters(),
        lr=EX_CONFIG["learning_rate"],
        weight_decay=EX_CONFIG["weight_decay"]
    )

    total_steps   = len(data) // EX_CONFIG["grad_accum_steps"]
    warmup_steps  = int(EX_CONFIG["warmup_ratio"] * total_steps)

    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=EX_CONFIG["scheduler_start_factor"],
        total_iters=warmup_steps
    )

    connector.train()
    optimizer.zero_grad()

    train_losses, train_accs, val_losses, val_accs, train_token_counts, val_token_counts = model_training(
        n_epochs=EX_CONFIG["epochs"],
        nano_chimera=nano_chimera,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        scheduler=scheduler,
        warmup_steps=warmup_steps,
        model_name="best_nano_chimera.pt"
    )
else:
    logger.info("PERFORM_STAGE1_TRAINING = False → skipping Stage 1.")

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
path = f"nano_chimera_connector_{ts}.pt"
torch.save(connector.state_dict(), path)

#### Inference!

Not only inference but we are going to add approximated translation Top-K to see the most relevant textual tokens associated with each visual token to get an intution of what is going on under the hood. TLDR: its not straight forward :)

In [ ]:
@torch.no_grad()
def get_visual_token_debug(
    llm,
    tokenizer,
    inputs_embeds,
    image_token_idx,
    batch_id=0,
    top_k=5,
):
    text_embeds = llm.get_input_embeddings().weight  # [V, D]
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

    visual_embeds = inputs_embeds[batch_id, image_token_idx]  # [K, D]
    visual_embeds = visual_embeds / visual_embeds.norm(dim=-1, keepdim=True)

    sims = visual_embeds @ text_embeds.T  # [K, V]
    topk = sims.topk(top_k, dim=-1)

    debug = {}
    for k in range(len(image_token_idx)):
        debug[k] = tokenizer.convert_ids_to_tokens(
            topk.indices[k].tolist()
        )
    return debug

In [ ]:
@torch.no_grad()
def inference_nano_chimera(
    nano_chimera,
    dataloader,
    tokenizer,
    connector,
    K,
    device,
    max_new_tokens=None,
    min_new_tokens=None,
    do_sample=None,
    temperature=None,
    top_p=None,
    top_k_visual_tokens=5,
    repetition_penalty=None,
    no_repeat_ngram_size=None,
    text_prompt=None  # New parameter for custom text string
):
    nano_chimera.eval()
    results = []

    # Grab standard defaults from the model config
    config = nano_chimera.llm.config
    max_new_tokens = max_new_tokens if max_new_tokens is not None else getattr(config, "max_new_tokens", 20)
    min_new_tokens = min_new_tokens if min_new_tokens is not None else getattr(config, "min_new_tokens", 0)
    do_sample = do_sample if do_sample is not None else getattr(config, "do_sample", False)
    temperature = temperature if temperature is not None else getattr(config, "temperature", 1.0)
    top_p = top_p if top_p is not None else getattr(config, "top_p", 1.0)
    repetition_penalty = repetition_penalty if repetition_penalty is not None else getattr(config, "repetition_penalty", 1.0)
    no_repeat_ngram_size = no_repeat_ngram_size if no_repeat_ngram_size is not None else getattr(config, "no_repeat_ngram_size", 0)

    # Set the default text prompt or use the custom text prompt provided
    if text_prompt is None:
        text_prompt = "<image> Please describe the contents of this image in one sentence."
    else:
        text_prompt = f"<image> {text_prompt}"

    for batch in dataloader:
        images = batch["image"]
        captions = batch["caption"]

        # Merge text + visual embeddings using the provided text prompt
        inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
            images=images,
            K=K,
            device=device,
            llm=nano_chimera.llm,
            tokenizer=tokenizer,
            connector=connector,
            texts=[text_prompt] * len(images)
            # Ensure the prompt is applied to all images in the batch
        )

        visual_debug = [
            get_visual_token_debug(
                llm=nano_chimera.llm,
                tokenizer=tokenizer,
                inputs_embeds=inputs_embeds,
                image_token_idx=image_token_idx,
                batch_id=i,
                top_k=top_k_visual_tokens,
            )
            for i in range(len(images))
        ]
        # Generate
        generated_ids = nano_chimera.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )

        # Decode predictions
        for i in range(len(images)):
            pred_caption = tokenizer.decode(generated_ids[i], skip_special_tokens=True)
            results.append({
                "image": images[i],
                "caption": captions[i],
                "generated": pred_caption,
                "visual_token_debug": visual_debug[i],
            })

    return results

In [ ]:
if PERFORM_INFERENCE:
    results = inference_nano_chimera(
        nano_chimera=nano_chimera,
        dataloader=test_loader,
        tokenizer=tokenizer,
        connector=connector,
        K=EX_CONFIG["n_visual_tokens"],
        device=DEVICE,
        max_new_tokens=50,
        min_new_tokens=10,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k_visual_tokens=5,
        repetition_penalty=1.2,
        no_repeat_ngram_size=2,
        text_prompt="Describe this image in one sentence."
    )
else:
    logger.info("PERFORM_INFERENCE = False → skipping inference.")

In [ ]:
if PERFORM_INFERENCE:
    for r in results[0:30]:
        print("=" * 80)
        print("GT :", r["caption"])
        print("GEN:", r["generated"])
        print("-" * 80)

        img = r["image"].permute(1, 2, 0).cpu().numpy()
        plt.imshow(img)
        plt.axis("off")
        plt.show()

        print("-" * 80)
        print("VISUAL TOKENS → NEAREST TEXT TOKENS")

        for k, tokens in r["visual_token_debug"].items():
            print(f"[token {k:02d}] {', '.join(tokens)}")

        print("=" * 80)
        print()

### 2.6 Pipeline Extension

In typical VLM training scenarios, analogously to LLM training, there are multiple phases not only a single iteration loop. For LLMs this is typically decomposed in the stages:

1. Pretraining: Similar to our training, the model learns to predict next tokens from **unsupervised corpora**, learning representations of language along the way.

2. Supervised Fine-Tuning: Through a **golden standard supervised Q&A dataset** the model learns to be a helpful assistant and to follow instructions.

3. Preference alingment (Reinforcement Learning from Human Feedback): Further refinement with Q&A style data with RL based algorithms like PPO, DPO, GRPO...

> However in multimodal VLM's the pipeline usually goes as following:

1. **Stage 1: Vision-Language Feature Alignment (Pre-training)**: In this phase the objective is align visual features from Visual Encoders with Language Model embeddings. Typical configurations of hyperparameters are:

    1-2 epochs on ~600K image-text pairs Batch Size: 256-512, Learning Rate: 1e-3 to 2e-3 (Bigger)

2. **Stage 2: Visual Instruction Tuning (SFT):**
  In this phase the objective is to teach the model to follow multimodal instructions. Typical configurations of hyperparameters are:

    3-5 epochs on ~150K instruction-examples Batch Size: 128-256 Learning Rate: 2e-5 (much smaller).




So if we want to refine the quality of our model to go beyond a simple pretraining modality alignment, we need to perform yet another training using visual instruction tuning datasets. Luckily for us again, huggingface contains such already processed datasets. For example: liuhaotian/LLaVA-Instruct-150K.


In [ ]:
if PERFORM_VISFT:
    VSFT_CONFIG = {
        # Data
        "train_size":     16,
        "test/val_ratio": 0.5,
        "batch_size":     16,

        # Optimization
        "learning_rate":  2e-5,
        "weight_decay":   0.01,

        # Connector
        "n_visual_tokens":       32,
        "connector_hidden_dims": (4096,),

        # Training
        "epochs":                 3,
        "grad_accum_steps":       8,
        "warmup_ratio":           0.05,
        "max_grad_norm":          1.0,
        "scheduler_start_factor": 0.1,

        # Logging
        "log_every":  128,
        "eval_every": 256,
    }
else:
    logger.info("PERFORM_VISFT = False --> skipping SFT config.")

In [ ]:
if PERFORM_VISFT:
    class LLaVASFTDataset(Dataset):
        def __init__(self, data_list, transform=None):
            self.data = data_list
            self.transform = transform

        def __getitem__(self, idx):
            item = self.data[idx]
            image_data = item["image"]
            caption    = item["caption"]

            if isinstance(image_data, str):
                try:
                    image = Image.open(image_data).convert("RGB")
                except (FileNotFoundError, Exception) as e:
                    raise ValueError(f"Failed to open image {image_data}") from e
            else:
                image = image_data

            if self.transform is not None:
                image = self.transform(image)

            return {"image": image, "caption": caption}

        def __len__(self):
            return len(self.data)

    # --- load & process LLaVA-Instruct-150K ---
    LLAVA_SFT_DATASET_NAME = "liuhaotian/LLaVA-Instruct-150K"
    print(f"Loading {LLAVA_SFT_DATASET_NAME} for SFT...")

    processed_sft_data = []
    dataset_stream_sft = load_dataset(
        LLAVA_SFT_DATASET_NAME, split="train", streaming=True
    ).shuffle(seed=SEED, buffer_size=1000)

    count = 0
    for row in dataset_stream_sft:
        if count >= VSFT_CONFIG["train_size"]:
            break
        image = row["image"]
        caption_parts = []
        has_image_instruction = False

        for conv in row["conversations"]:
            if conv["from"] == "human":
                if "<image>" in conv["value"]:
                    has_image_instruction = True
                question_text = conv["value"].replace("<image>", "").strip()
                if question_text:
                    caption_parts.append(f"User: {question_text}")
            elif conv["from"] == "gpt" and conv["value"]:
                caption_parts.append(f"Assistant: {conv['value']}")

        final_caption = " ".join(caption_parts).strip()
        if image and final_caption and has_image_instruction:
            processed_sft_data.append({"image": image, "caption": final_caption})
            count += 1

    print(f"Processed {len(processed_sft_data)} SFT samples.")

    full_sft_dataset = LLaVASFTDataset(processed_sft_data)

    sft_train_ratio = 1 - VSFT_CONFIG["test/val_ratio"]
    sft_val_ratio   = VSFT_CONFIG["test/val_ratio"] / 2
    sft_test_ratio  = VSFT_CONFIG["test/val_ratio"] / 2

    n_total_sft  = len(full_sft_dataset)
    n_sft_train  = int(sft_train_ratio * n_total_sft)
    n_sft_val    = int(sft_val_ratio   * n_total_sft)
    n_sft_test   = n_total_sft - n_sft_train - n_sft_val

    generator = torch.Generator().manual_seed(SEED)
    sft_train_ds, sft_val_ds, sft_test_ds = random_split(
        full_sft_dataset, [n_sft_train, n_sft_val, n_sft_test], generator=generator
    )

    sft_train_loader = DataLoader(sft_train_ds, batch_size=VSFT_CONFIG["batch_size"], shuffle=True,  collate_fn=collate_fn)
    sft_val_loader   = DataLoader(sft_val_ds,   batch_size=VSFT_CONFIG["batch_size"], shuffle=True,  collate_fn=collate_fn)
    sft_test_loader  = DataLoader(sft_test_ds,  batch_size=VSFT_CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    print("SFT DataLoaders ready.")

In [ ]:
if PERFORM_VISFT:
    # TODO: LLaVA-Instruct-150K images are filenames, not actual images.
    #       Find a dataset with real embedded images before running this.

    criterion_sft = nn.CrossEntropyLoss().to(DEVICE)

    optimizer_sft = torch.optim.AdamW(
        connector.parameters(),
        lr=VSFT_CONFIG["learning_rate"],
        weight_decay=VSFT_CONFIG["weight_decay"]
    )

    total_steps_sft  = len(full_sft_dataset) // VSFT_CONFIG["grad_accum_steps"]
    warmup_steps_sft = int(VSFT_CONFIG["warmup_ratio"] * total_steps_sft)

    scheduler_sft = torch.optim.lr_scheduler.LinearLR(
        optimizer_sft,
        start_factor=VSFT_CONFIG["scheduler_start_factor"],
        total_iters=warmup_steps_sft
    )

    connector.train()
    optimizer_sft.zero_grad()

    (train_losses_sft, train_accs_sft,
     val_losses_sft,   val_accs_sft,
     train_token_counts_sft, val_token_counts_sft) = model_training(
        n_epochs=VSFT_CONFIG["epochs"],
        nano_chimera=nano_chimera,
        train_loader=sft_train_loader,
        val_loader=sft_val_loader,
        optimizer=optimizer_sft,
        criterion=criterion_sft,
        device=DEVICE,
        scheduler=scheduler_sft,
        warmup_steps=warmup_steps_sft,
        model_name="best_nano_chimera_sft.pt"
    )
else:
    logger.info("PERFORM_VISFT = False → skipping SFT training.")

In [ ]:
if PERFORM_VISFT:
    print(full_sft_dataset.data[0])

In [ ]:
# (pipeline definitions live above; nothing to run here)

## 3. Experiments & Tracking

In this section we are going to execute some experiments to try to optimize this NanoChimera architecture as much as possible with different configurations of hyperparameters and even small architecture size changes. Given this multi-variable optimization landscape, we need to take into account all of the hyperparameters and factors that can affect the model performance.

---

This section purpose is simply experiment tracking similar to Weights&Biases, MLFlow or other MLOps tools.

## 3.0 Basic Experiment 1

**Hyperparameters**  
- Dataset size: 4096  
- Batch size: 32  
- Epochs: 2  
- Learning Rate: 9e-4  
- Number of visual tokens: 32  
- Connector shape: 4096  

---

Results

- Validation Loss: 3.32
- Mean Token Accuracy: 0.601
- Perplexity: 27.78
- Total Number of Tokens: 186578

## 3.1 Basic Experiment 2

**Hyperparameters**  
- Dataset size: 16384  
- Batch size: 32  
- Gradient Accumulation Step: 8
- Epochs: 4
- Learning Rate: 9e-4  
- Number of visual tokens: 32  
- Connector shape: 4096  

---

Results

- Validation Loss: 2.828
- Mean Token Accuracy: 0.6039
- Perplexity: 16.92
- Total Number of Tokens: 1.45M

### 3.2 Basic Experiment 3

**Hyperparameters**  
- Dataset size: 32768  
- Batch size: 16
- Gradient Accumulation Step: 8  
- Epochs: 4
- Learning Rate: 1e-5
- Number of visual tokens: 32  
- Connector shape: 4096  

---

Results

- Validation Loss: 3.458
- Mean Token Accuracy: 0.566
- Perplexity: 31.76
- Total Number of Tokens: 2.67M

### 3.3 Basic Experiment 4

**Hyperparameters**  
- Dataset size: 40000
- Batch size: 32
- Gradient Accumulation Step: 2  
- Epochs: 3
- Learning Rate: 1e-5
- Number of visual tokens: 32  
- Connector shape: (4096, 2048)

---

- Validation Loss: 3.523
- Mean Token Accuracy: 0.6587
- Perplexity: 33.90
- Total Number of Tokens: 2.23M



### 3.4 Basic Experiment 5

**Hyperparameters**  
- Dataset size: 40000
- Batch size: 32
- Gradient Accumulation Step: 2  
- Epochs: 6
- Learning Rate: 1e-5
- Number of visual tokens: 32  
- Connector shape: 6000

---

Results

- Validation Loss: 3.016
- Mean Token Accuracy: 0.622
- Perplexity: 20.41
- Total Number of Tokens: 5.67M


### 3.5 Basic Experiment 6

**Hyperparameters**  
- Dataset size: 40000
- Batch size: 32
- Gradient Accumulation Step: 2  
- Epochs: 6
- Learning Rate: 1e-5
- Number of visual tokens: 40  
- Connector shape: 6000

---

Results

- Validation Loss: 2.973
- Mean Token Accuracy: 0.632
- Perplexity: 19.87
- Total Number of Tokens: 5.67M

### 3.6 Basic Experiment 7

**Hyperparameters**  
- Dataset size: 40000
- Batch size: 28
- Gradient Accumulation Step: 2  
- Epochs: 6
- Learning Rate: 1e-5
- Number of visual tokens: 42  
- Connector shape: 4096

---

Results

- Validation Loss: 3.066
- Mean Token Accuracy: 0.613
- Perplexity: 21.46
- Total Number of Tokens: 5.62M


### 3.7 Basic Experiment 8

**Hyperparameters**  
- Dataset size: 64000
- Batch size: 58
- Gradient Accumulation Step: 1
- Epochs: 9
- Learning Rate: 3e-5
- Number of visual tokens: 32  
- Connector shape: 4096

---

Results

- Validation Loss: 2.529
- Mean Token Accuracy: 0.682
- Perplexity: 12,55
- Total Number of Tokens: 12.31M

### 3.8 Basic Experiment 9

**Hyperparameters**  
- Dataset size: 70000
- Batch size: 58
- Gradient Accumulation Step: 1
- Epochs: 8
- Learning Rate: 3e-5
- Number of visual tokens: 37
- Connector shape: 4096

---

Results

- Validation Loss: 2.456
- Mean Token Accuracy: 0.689
- Perplexity: 11.66
- Total Number of Tokens: 14.25M



- TOTAL NUMBER OF TOKENS: 50M
- KING JAMES BIBLE: 1M
- THE WHOLE SHAKESPEAR COLLECTION: 33M

### 3.9 Basic Experiment 1 (locally)

**Hyperparameters**  
- Dataset size: 8192
- Batch size: 32
- Gradient Accumulation Step: 2  
- Epochs: 6
- Learning Rate: 1e-5
- Number of visual tokens: 32  
- Connector shape: (4096, 2048)

---

Results

- Validation Loss: 3.487
- Mean Token Accuracy: 0.600
- Perplexity: 32.68
- Total Number of Tokens: 164028

### 3.10 Basic Experiment 2 (locally)

**Hyperparameters**  
- Dataset size: 8192
- Batch size: 32
- Gradient Accumulation Step: 2  
- Epochs: 6
- Learning Rate: 1e-5
- Number of visual tokens: 32  
- Connector shape: (4096, 2048)

---

Results

- Validation Loss: 3.487
- Mean Token Accuracy: 0.600
- Perplexity: 32.68
- Total Number of Tokens: 164028

## 4. Evaluation pipeline


In this section, we evaluate the performance of our multimodal model using **task-specific benchmarks** rather than relying solely on generic metrics like loss or perplexity.  

While metrics such as training/validation loss or perplexity are useful for monitoring model convergence, they **do not fully capture the model's reasoning abilities or real-world performance** on multimodal tasks. For example, a model might achieve low loss but still fail to answer visual questions correctly or reason about images in context.  

To address this, we focus on the following benchmarks:  

- **MMBench**: Tests general multimodal reasoning, commonsense understanding, and scene comprehension.  
- **VQAv2**: Standard benchmark for visual question answering, assessing object recognition, counting, and attribute understanding.  

By evaluating on these benchmarks, we obtain a **more accurate and practical measure of the model's capabilities**, aligning our results with published baselines such as LLaVA.


Although there's no silver bullet benchmark, most of them are curated and offer high quality assessing capabilities of AI systems. **lm_eval_harness** has become the standard de-facto evaluation system for Language models (LM) , similarly this framework **lmms_eval** is becoming also the de-facto for multimodal language models (MLM)

In [ ]:
# GENERAL LLMS EVAL HELP
#!lmms-eval --help
# THIS SERVES TO LIST THE EVALUATION TASKS AVAILABLE (MULTIMODAL TASKS)
#!python -m lmms_eval --tasks list

In [ ]:
class NanoChimeraLMMSWrapper(lmms):
    """
    LMMS eval model wrapper for NanoChimera-style models.
    Implements minimal LMMS expected methods in a robust, notebook-friendly way.
    """
    def __init__(
        self,
        nano_chimera,
        tokenizer,
        vision_processor,
        vision_model,
        connector,
        n_visual_tokens: int,
        device: Optional[torch.device] = None
    ):
        super().__init__()
        self.nano_chimera       = nano_chimera
        self.tokenizer          = tokenizer
        self.vision_processor   = vision_processor
        self.vision_model       = getattr(vision_model, "vision_model", vision_model)
        self.connector          = connector
        self.n_visual_tokens    = int(n_visual_tokens)
        self.device             = device or getattr(nano_chimera.llm, "device", torch.device("cpu"))
        self.is_vllm            = False
        self.max_length         = 2048
        self.max_gen_toks       = 32

    # --------------------------------------------------
    # LMMS-compatible entrypoints
    # --------------------------------------------------
    def generate_until(self, requests: list[Instance]) -> list[str]:
        outputs = []
        for inst in requests:
            question       = inst.args[0]
            gen_kwargs     = inst.args[1].copy()
            doc_to_visual  = inst.args[2]

            image = None
            if inst.doc is not None:
                visuals = doc_to_visual(inst.doc)
                image = visuals[0] if visuals else None

            max_new_tokens = gen_kwargs.pop("max_new_tokens", self.max_gen_toks)
            answer = self.generate_answer(
                image=image,
                prompt=question,
                max_new_tokens=max_new_tokens,
                **{k: v for k, v in gen_kwargs.items() if k != "until"}
            )
            outputs.append(answer)
        return outputs

    def generate_until_multi_round(self, requests: List[Any]) -> List[List[Dict[str, str]]]:
        return [self.generate_until([req]) for req in requests]

    def loglikelihood(self, requests: List[Instance]) -> List[tuple[float, bool]]:
        return [(0.0, False) for _ in requests]  # placeholder

    # --------------------------------------------------
    # Core generation
    # --------------------------------------------------
    def generate_answer(self, image, prompt: str, max_new_tokens: int = 20, **kwargs) -> str:
        self.nano_chimera.eval()

        if image is None:
            return "[no image provided; placeholder answer]"

        # vision encoding
        inputs = self.vision_processor(images=image, return_tensors="pt")
        vision_device = next(self.vision_model.parameters()).device
        inputs = {k: v.to(vision_device) for k, v in inputs.items()}

        with torch.no_grad():
            vis_out  = self.vision_model(**inputs)
            vis_feats = getattr(vis_out, "last_hidden_state", vis_out)[..., :self.n_visual_tokens, :]

        connector_device = getattr(self.connector, "device", vision_device)
        vis_feats = vis_feats.to(connector_device)
        with torch.no_grad():
            vis_embeds = self.connector(vis_feats)

        # text encoding
        text_input = f"{IMAGE_TOKEN} {prompt}"
        tokenized  = self.tokenizer(text_input, return_tensors="pt")
        input_ids  = tokenized.input_ids.to(self.device)

        image_token_id = self.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
        embed_layer    = self.nano_chimera.llm.get_input_embeddings()
        text_embeds    = embed_layer(input_ids).to(vis_embeds.dtype)

        idx_positions = (input_ids == image_token_id).nonzero(as_tuple=False)
        if idx_positions.shape[0] == 0:
            raise ValueError(f"IMAGE_TOKEN ({IMAGE_TOKEN}) not found in tokenized input.")
        idx = idx_positions[0, 1].item()

        left  = text_embeds[:, :idx, :]
        right = text_embeds[:, idx + 1:, :]

        if vis_embeds.ndim == 2:
            vis_embeds = vis_embeds.unsqueeze(0)
        vis_embeds = vis_embeds.to(text_embeds.dtype)

        input_embeds  = torch.cat([left, vis_embeds, right], dim=1).to(self.device)
        attention_mask = torch.ones(input_embeds.size()[:-1], device=self.device, dtype=torch.long)

        generation_kwargs = {
            "inputs_embeds":  input_embeds,
            "attention_mask": attention_mask,
            "max_new_tokens": max_new_tokens,
            "pad_token_id":   getattr(self.tokenizer, "pad_token_id", self.tokenizer.eos_token_id),
            "do_sample":      False,
            "temperature":    0.0,
            **kwargs,
        }

        with torch.no_grad():
            generated_ids = self.nano_chimera.llm.generate(**generation_kwargs)

        gen_text = (self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                    if isinstance(generated_ids, torch.Tensor) else str(generated_ids))

        if text_input.strip() in gen_text:
            gen_text = gen_text.replace(text_input.strip(), "", 1).strip()

        return gen_text.strip()

# --------------------------------------------------
# Wrapper instantiation + task setup (gated)
# --------------------------------------------------
if PERFORM_EVALUATION:
    lmms_model_wrapper = NanoChimeraLMMSWrapper(
        nano_chimera=nano_chimera,
        tokenizer=tokenizer,
        vision_processor=vision_processor,
        vision_model=vision_model,
        connector=connector,
        n_visual_tokens=EX_CONFIG["n_visual_tokens"],
        device=DEVICE
    )

    lmms_eval_config = {
        "task_names":  ["llava_in_the_wild"],
        "num_fewshot": 0,
        "batch_size":  1,
        "limit":       4,
        "verbose":     True,
        "predict_only": True,
    }

    tm = TaskManager()
    tasks = tm.load_task_or_group(["llava_in_the_wild"], task_type="simple")
    tasks["llava_in_the_wild"].config.full_docs = True
else:
    logger.info("PERFORM_EVALUATION = False → skipping evaluation setup.")

In [ ]:
if PERFORM_EVALUATION:
    lmms_evaluation_results = simple_evaluate(
        batch_size=lmms_eval_config["batch_size"],
        numpy_random_seed=SEED,
        random_seed=SEED,
        torch_random_seed=SEED,
        fewshot_random_seed=SEED,
        model=lmms_model_wrapper,
        tasks=lmms_eval_config["task_names"],
        num_fewshot=lmms_eval_config["num_fewshot"],
        limit=lmms_eval_config["limit"],
        predict_only=lmms_eval_config["predict_only"],
        model_args={"hf_token": os.environ.get("HUGGINGFACE_TOKEN")},
    )
    print("LMMS Evaluation Results:")
    print(lmms_evaluation_results)
else:
    logger.info("PERFORM_EVALUATION = False → skipping simple_evaluate.")

In [ ]:
if PERFORM_EVALUATION:
    tm_debug = TaskManager("INFO")
    task_dict = get_task_dict(["llava_in_the_wild"], tm_debug, task_type="simple")
    task = list(task_dict.values())[0]
    task.build_all_requests(limit=5)

    inst = task.instances[3]
    print("TYPE:",        type(inst))
    print("IS Instance:", isinstance(inst, Instance))
    print("ARGS:",        inst.args)
    print("HAS DOC:",     inst.doc is not None)
    print("DOC KEYS:",    inst.doc.keys())
    print("REQUEST TYPE:",inst.request_type)

In [ ]:
if PERFORM_EVALUATION:
    dummy_image  = Image.new("RGB", (224, 224), color="white")
    dummy_image2 = Image.new("RGB", (224, 224), color="gray")

    mmbench_dummy = [
        {"question": "What is the object in the scene?", "answer": "cat",  "image": dummy_image},
        {"question": "What color is the car?",           "answer": "red",  "image": dummy_image2},
    ]
    vqav2_dummy = [
        {"question": "How many cats?",          "answer": "2",    "image": dummy_image},
        {"question": "What color is the sky?",  "answer": "blue", "image": dummy_image2},
    ]

    dummy_eval_config = {
        "task_names": ["MMBench", "VQAv2"],
        "num_fewshot": 0,
        "limit":    2,
        "verbose":  True,
        "save_path": "./lmms_results.json"
    }

    lmms_evaluation_results = {}
    for task_name, dummy_dataset in zip(dummy_eval_config["task_names"], [mmbench_dummy, vqav2_dummy]):
        print(f"\n=== Evaluating {task_name} ===")
        task_results = []
        for i, example in enumerate(dummy_dataset):
            output = lmms_model_wrapper.generate_answer(
                image=example["image"], prompt=example["question"]
            )
            task_results.append({
                "question":    example["question"],
                "ground_truth": example["answer"],
                "generated":   output
            })
            if dummy_eval_config["verbose"]:
                print(f"  [{i+1}/{len(dummy_dataset)}] {example['question']}")
                print(f"  Generated: {output}")
        lmms_evaluation_results[task_name] = task_results

    with open(dummy_eval_config["save_path"], "w") as f:
        json.dump(lmms_evaluation_results, f, indent=2)

    print(f"\nEvaluation saved to {dummy_eval_config['save_path']}")
    print(json.dumps(lmms_evaluation_results, indent=2))

In [ ]:
if PERFORM_EVALUATION:
    ll_results = lmms_model_wrapper.loglikelihood([
        {"prompt": "How many cats?", "image": dummy_image, "label": "2"}
    ])
    print(ll_results)

## 5. Results

In this section we are going to discuss the different experiment results, as well as the winning model pipeline combination.

Also we are going to upload it to hugging-face to opensource it for everybody to use freely, although it lacks some refinement.



In [ ]:
# 1. PRESENT SINGLE-RESULTS TABLE
# Flatten results into a single DF
all_rows = []
for task, examples in lmms_evaluation_results.items():
    for ex in examples:
        all_rows.append({
            "Task": task,
            "Question": ex["question"],
            "Ground Truth": ex["ground_truth"],
            "Generated": ex["generated"]
        })

df = pd.DataFrame(all_rows)
df



In [ ]:
# 2, PRESENT AGGREGATED RESULTS TABLE
summary = {}
for task, examples in lmms_evaluation_results.items():
    correct = sum(ex["ground_truth"].lower() == ex["generated"].lower() for ex in examples)
    total = len(examples)
    summary[task] = {"accuracy": correct / total}

summary_df = pd.DataFrame(summary).T
summary_df


In [ ]:
if PUSH_TO_HUB:
    LLM_NAME_SHORT    = LLM_NAME.split("/")[-1].split("-")[0]
    VISION_NAME_SHORT = VISION_NAME.split("/")[-1].split("-")[0]
    repo_name = f"NanoChimera-{LLM_NAME_SHORT}-{VISION_NAME_SHORT}-aligned"

    nano_chimera.llm.push_to_hub(repo_name)
    tokenizer.push_to_hub(repo_name)

    connector_path = "connector_weights.pt"
    torch.save(connector.state_dict(), connector_path)
    huggingface_hub.upload_file(
        path_or_fileobj=connector_path,
        path_in_repo=connector_path,
        repo_id=repo_name
    )
    print(f"Model pushed to huggingface.co/{repo_name}")
else:
    logger.info("PUSH_TO_HUB = False → skipping HuggingFace push.")

 ## 6. Conclusions